In [1]:
import logging
import numpy as np
import math
import os
import sys
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Any, Mapping
from pathlib import Path
import datasets
import json
import torch
from build_dataset import buid_instruction_dataset, DataCollatorForSupervisedDataset
import transformers
from transformers import (
    CONFIG_MAPPING,
    MODEL_FOR_CAUSAL_LM_MAPPING,
    AutoConfig,
    AutoModelForCausalLM,
    LlamaForCausalLM,
    LlamaTokenizer,
    AutoTokenizer,
    HfArgumentParser,
    Trainer,
    TrainingArguments,
    set_seed,
)

from transformers.trainer_utils import get_last_checkpoint
from transformers.utils import check_min_version, send_example_telemetry
from transformers.utils.versions import require_version

from peft import LoraConfig, TaskType, get_peft_model, PeftModel, get_peft_model_state_dict




IGNORE_INDEX = -100
DEFAULT_PAD_TOKEN = "[PAD]"
DEFAULT_EOS_TOKEN = "</s>"
DEFAULT_BOS_TOKEN = "<s>"
DEFAULT_UNK_TOKEN = "<unk>"

# Will error if the minimal version of Transformers is not installed. Remove at your own risks.
# check_min_version("4.28.0.dev0")

require_version("datasets>=1.8.0", "To fix: pip install -r examples/pytorch/language-modeling/requirements.txt")



===================================BUG REPORT===================================
Welcome to bitsandbytes. For bug reports, please run

python -m bitsandbytes

 and submit this information together with your error trace to: https://github.com/TimDettmers/bitsandbytes/issues
bin /root/miniconda3/envs/pytorch2-0/lib/python3.10/site-packages/bitsandbytes/libbitsandbytes_cuda120.so
CUDA SETUP: CUDA runtime path found: /usr/local/cuda-12.0/lib64/libcudart.so
CUDA SETUP: Highest compute capability among GPUs detected: 8.9
CUDA SETUP: Detected CUDA version 120
CUDA SETUP: Loading binary /root/miniconda3/envs/pytorch2-0/lib/python3.10/site-packages/bitsandbytes/libbitsandbytes_cuda120.so...


/root/miniconda3/envs/pytorch2-0/lib/python3.10/site-packages/bitsandbytes/cuda_setup/main.py:149: UserWarning: /root/miniconda3/envs/pytorch2-0 did not contain ['libcudart.so', 'libcudart.so.11.0', 'libcudart.so.12.0'] as expected! Searching further paths...
  warn(msg)


In [2]:
# ModelArguments

model_name_or_path = "/data/models/llama-alpaca-plus-7B"

tokenizer_name_or_path = "/root/SourceCode/DL/Chinese-LLaMA-Alpaca/scripts/merged_tokenizer_sp/chinese_llama.model"

torch_dtype = torch.float16

In [3]:
# DataTrainingArguments

dataset_dir = '/root/SourceCode/DL/Chinese-LLaMA-Alpaca/data'
validation_file = '/root/SourceCode/DL/Chinese-LLaMA-Alpaca/scripts/validation_file_name/vaidation.json'
preprocessing_num_workers = 8


lr=1e-4
lora_rank=8
lora_alpha=32
lora_trainable="q_proj,v_proj,k_proj,o_proj,gate_proj,down_proj,up_proj"
# modules_to_save="embed_tokens,lm_head"
lora_dropout=0.05
max_seq_length=512
cache_dir = None
model_revision = "main"
output_dir = "/data/chinese-llama/chinese_alpaca_plus_lora_7b_gitlab2"

training_args = TrainingArguments(
    output_dir=output_dir,
    overwrite_output_dir=True,
    max_steps=2000,)
set_seed(training_args.seed)
training_args

TrainingArguments(
_n_gpu=1,
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_pin_memory=True,
ddp_backend=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=False,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_steps=None,
evaluation_strategy=no,
fp16=False,
fp16_backend=auto,
fp16_full_eval=False,
fp16_opt_level=O1,
fsdp=[],
fsdp_config={'fsdp_min_num_params': 0, 'xla': False, 'xla_fsdp_grad_ckpt': False},
fsdp_min_num_params=0,
fsdp_transformer_layer_cls_to_wrap=None,
full_determinism=False,
gradient_accumulation_steps=1,
gradient_checkpointing=False,
greater_is_better=None,
group_by_length=False,
half_precision_backend=auto,
hub_model_id=None,
hub_private_repo=False,
hub_strategy=every_save,
hub_token=

In [4]:
config = AutoConfig.from_pretrained(model_name_or_path)
tokenizer = LlamaTokenizer.from_pretrained(model_name_or_path)

## 数据处理

In [5]:
# 训练数据集

data_collator = DataCollatorForSupervisedDataset(tokenizer=tokenizer)

path = Path(dataset_dir)
files = [os.path.join(path,file.name) for file in path.glob("*.json")]
print(f"training files: {' '.join(files)}")
train_dataset = buid_instruction_dataset(
    data_path=files, 
    tokenizer=tokenizer, 
    max_seq_length=max_seq_length,
    data_cache_dir = None, 
    preprocessing_num_workers = preprocessing_num_workers)

training files: /root/SourceCode/DL/Chinese-LLaMA-Alpaca/data/gitlab_QA_zh.json


In [6]:
# 验证数据集

files = [validation_file]
print(f"training files: {' '.join(files)}")
eval_dataset = buid_instruction_dataset(
    data_path=files, 
    tokenizer=tokenizer, 
    max_seq_length=max_seq_length,
    data_cache_dir = None, 
    preprocessing_num_workers = preprocessing_num_workers)

training files: /root/SourceCode/DL/Chinese-LLaMA-Alpaca/scripts/validation_file_name/vaidation.json


## 使用peft 加载lora模型

In [7]:
# 加载基础模型

model = LlamaForCausalLM.from_pretrained(
    model_name_or_path,
    from_tf=bool(".ckpt" in model_name_or_path),
    config=config,
    cache_dir=cache_dir,
    revision=model_revision,
    use_auth_token=None,
    torch_dtype=torch_dtype,
    low_cpu_mem_usage=True
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [8]:
embedding_size = model.get_input_embeddings().weight.shape[0]
embedding_size

49954

In [9]:
len(tokenizer)

49954

In [10]:
target_modules = lora_trainable.split(',')

peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    target_modules=target_modules,
    inference_mode=False, 
    r=lora_rank, lora_alpha=lora_alpha, 
    lora_dropout=lora_dropout,
    modules_to_save=None)

In [11]:
model = get_peft_model(model, peft_config)

In [12]:
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(49954, 4096, padding_idx=0)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): Linear(
                in_features=4096, out_features=4096, bias=False
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
              )
              (k_proj): Linear(
                in_features=4096, out_features=4096, bias=False
        

In [13]:
model.print_trainable_parameters()

trainable params: 19988480 || all params: 6905483264 || trainable%: 0.28945809056123434


In [14]:
old_state_dict = model.state_dict
old_state_dict

<bound method Module.state_dict of PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(49954, 4096, padding_idx=0)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): Linear(
                in_features=4096, out_features=4096, bias=False
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
              )
              (k_proj): Linear(
                in_features=4096, out

In [15]:
trainer = Trainer(
        args=training_args,
        model=model,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        tokenizer=tokenizer,
        data_collator=data_collator,
    )

In [16]:
output_dir = "/data/chinese-llama/chinese_alpaca_plus_lora_7b_gitlab2"
last_checkpoint = get_last_checkpoint(output_dir)

In [17]:
checkpoint = last_checkpoint
train_result = trainer.train(resume_from_checkpoint=checkpoint)
trainer.save_model()

/root/miniconda3/envs/pytorch2-0/lib/python3.10/site-packages/transformers/optimization.py:407: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Step,Training Loss
500,0.221100
1000,0.037100
1500,0.009700
2000,0.004400


In [18]:
metrics = train_result.metrics

In [19]:
metrics["train_samples"] = len(train_dataset)
trainer.save_metrics("train", metrics)
trainer.save_state()

In [20]:
metrics

{'train_runtime': 425.3235,
 'train_samples_per_second': 37.618,
 'train_steps_per_second': 4.702,
 'total_flos': 4.08207721525248e+16,
 'train_loss': 0.0680610259771347,
 'epoch': 13.79,
 'train_samples': 1158}

In [21]:
metrics = trainer.evaluate()
metrics["eval_samples"] =len(eval_dataset)

In [22]:
perplexity = math.exp(metrics["eval_loss"])

In [23]:
metrics["perplexity"] = perplexity


In [24]:
metrics

{'eval_loss': 1.0927734375,
 'eval_runtime': 1.0437,
 'eval_samples_per_second': 97.728,
 'eval_steps_per_second': 12.456,
 'epoch': 13.79,
 'eval_samples': 102,
 'perplexity': 2.98253448538564}

In [25]:
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(49954, 4096, padding_idx=0)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): Linear(
                in_features=4096, out_features=4096, bias=False
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
              )
              (k_proj): Linear(
                in_features=4096, out_features=4096, bias=False
        

In [44]:
input_text = "Gitlab 故障分为以下类型: pipeline流水线问题、代码质量、Runner问题、git操作及代码管理、API调用、第三方集成、集群架构及GEO、数据迁移、用户权限及认证、安装部署升级、审计、系统>异常\n请对如下问题进行分类：\n如何确定在Gitlab中做了什么更改?"
inputs = tokenizer(input_text,return_tensors="pt")
print(inputs)

generation_config = dict(
    temperature=0.1,
    top_k=40,
    top_p=0.9,
    do_sample=True,
    num_beams=1,
    repetition_penalty=1.3,
    max_new_tokens=400
    )

model.eval()
device = torch.device(0)
with torch.no_grad():
    generation_output = model.generate(
                    input_ids = inputs["input_ids"].to(device), 
                    attention_mask = inputs['attention_mask'].to(device),
                    eos_token_id=tokenizer.eos_token_id,
                    pad_token_id=tokenizer.pad_token_id,
                    **generation_config
                )
    print(generation_output)

{'input_ids': tensor([[    1, 11786,  8205, 29871, 36696, 33747, 32595, 33812, 29901, 16439,
         40406, 31532, 32026, 30330, 35116, 32578, 30330, 16802, 32026, 30330,
          5559, 32624, 31436, 29958, 35116, 32069, 30330,  8787, 31268, 30406,
         30330, 38495, 37618, 30330, 30893, 31829, 40476, 31436,  1692, 29949,
         30330, 32326, 44236, 30330, 32308, 44820, 31436, 35867, 30330, 33455,
         36953, 33656, 30330, 37718, 30330, 32077, 29958, 35365,    13, 31088,
         30783, 34001, 32026, 32019, 34727, 30383,    13, 32175, 32932, 30505,
         28712,  8205, 30275, 34008, 32066, 41699, 29973]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1]])}
tensor([[    1, 11786,  8205, 29871, 36696, 33747, 32595, 33812, 29901, 16439,
     

In [41]:
output = tokenizer.decode(generation_output[0],skip_special_tokens=True)
output = output.strip()
output

'Gitlab 故障分为以下类型: pipeline流水线问题、代码质量、Runner问题、git操作及>代码管理、API调用、第三方集成、集群架构及GEO、数据迁移、用户权限及认证、安装部署升级、审计、系统>异常\n请对如下问题进行分类：\n如何确定在Gitlab中做了什么更改?'